# Lesson 3 - Building a QA Agent with Gemini

In this lesson, you will build a basic Question Answering (QA) agent using Google's Gemini API. You will start by directly calling Gemini to analyze a PDF document containing health insurance policy details. Then, you will refactor this logic into a reusable Python class, laying the groundwork for the next lesson where you will wrap this agent in an Agent2Agent (A2A) server.

## 3.1. Import Libraries and Setup

First, import the necessary libraries. You will use the Gemini REST API so the notebook only needs a Gemini API key and standard Python libraries.

In [20]:
import base64
from pathlib import Path

import requests
from IPython.display import Markdown, display

from helpers import get_gemini_api_key, get_gemini_model, setup_env

In [18]:
%cd /content/drive/MyDrive/A2A-MCP

/content/drive/MyDrive/A2A-MCP


In [32]:
!pip install -r requirements.txt 

  Using cached a2a_sdk-0.3.26-py3-none-any.whl.metadata (9.2 kB)
  Using cached ag_ui_protocol-0.1.15-py3-none-any.whl.metadata (2.9 kB)
  Using cached agent_framework-1.0.0b251114-py3-none-any.whl.metadata (9.8 kB)
  Using cached agent_framework_a2a-1.0.0b260130-py3-none-any.whl.metadata (1.9 kB)
  Using cached agent_framework_ag_ui-1.0.0b260130-py3-none-any.whl.metadata (6.8 kB)
  Using cached agent_framework_anthropic-1.0.0b260130-py3-none-any.whl.metadata (1.7 kB)
  Using cached agent_framework_azure_ai-1.0.0b260130-py3-none-any.whl.metadata (1.5 kB)
  Using cached agent_framework_azurefunctions-1.0.0b260130-py3-none-any.whl.metadata (2.2 kB)
  Using cached agent_framework_chatkit-1.0.0b260130-py3-none-any.whl.metadata (6.5 kB)
  Using cached agent_framework_copilotstudio-1.0.0b260130-py3-none-any.whl.metadata (4.5 kB)
  Using cached agent_framework_core-1.0.0b251120-py3-none-any.whl.metadata (9.8 kB)
  Using cached agent_framework_devui-1.0.0b260130-py3-none-any.whl.metadata (17 k

In [16]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [21]:
setup_env()

For this workshop, the `.env` file only needs a Gemini API key and optional Gemini settings such as `GEMINI_MODEL`.

## 3.2. Initialize the Client and Load Data

Here you initialize the Gemini request settings using the API key from `.env`.


In [22]:
api_key = get_gemini_api_key()
model = get_gemini_model()

**Note:** This workshop uses the Gemini API directly through Google AI Studio style API keys, so you do not need Vertex AI credentials or Anthropic configuration.

In [23]:
api_url = (
    "https://generativelanguage.googleapis.com/v1beta/models/"
    f"{model}:generateContent"
)

You also read the insurance policy PDF (`2026AnthemgHIPSBC.pdf`) and encode it in base64 so it can be passed to the model as context. You can find the PDF document in the [course repo](https://github.com/holtskinner/A2AWalkthrough/tree/main/data) or in the `data` folder (please check the note at the beginning of the notebook on how to access the `data` folder).

In [24]:
with Path("./data/2026AnthemgHIPSBC.pdf").open("rb") as file:
    pdf_data = base64.standard_b64encode(file.read()).decode("utf-8")

## 3.3. Query the Model

Now you will send a specific query to the model: "How much would I pay for mental health therapy?". You provide Gemini with a system instruction to act as an expert insurance agent and pass the PDF document alongside the user's text prompt.

In [25]:
prompt = "How much would I pay for mental health therapy?"

In [26]:
response = requests.post(
    api_url,
    headers={
        "Content-Type": "application/json",
        "x-goog-api-key": api_key,
    },
    json={
        "system_instruction": {
            "parts": [
                {
                    "text": "You are an expert insurance agent designed to assist with coverage queries. Use the provided documents to answer questions about insurance policies. If the information is not available in the documents, respond with 'I don't know'."
                }
            ]
        },
        "generationConfig": {"temperature": 0.2},
        "contents": [
            {
                "parts": [
                    {
                        "inline_data": {
                            "mime_type": "application/pdf",
                            "data": pdf_data,
                        }
                    },
                    {"text": prompt},
                ]
            }
        ],
    },
    timeout=120,
)
response.raise_for_status()
result = response.json()

In [27]:
response_text = result["candidates"][0]["content"]["parts"][0]["text"].replace("$", r"\\$")
display(Markdown(response_text))

If you need mental health, behavioral health, or substance abuse services, you would pay 10% coinsurance for in-network outpatient office visits and 30% coinsurance for out-of-network outpatient office visits. For inpatient services, you would pay 10% coinsurance for in-network and 30% coinsurance for out-of-network.

## 3.4. Refactor into an Agent Class

To make this code reusable and easier to integrate into an A2A server later, you will wrap the logic into a `PolicyAgent` class in a file named `agents.py`. This class initializes the client and data in the `__init__` method and exposes an `answer_query` method.

In [28]:
%%writefile agents.py
import base64
from pathlib import Path

import requests

from helpers import get_gemini_api_key, get_gemini_model, setup_env

SYSTEM_INSTRUCTION = (
    "You are an expert insurance agent designed to assist with coverage "
    "queries. Use the provided documents to answer questions about insurance "
    "policies. If the information is not available in the documents, respond "
    "with 'I don't know'."
)

class PolicyAgent:
    def __init__(self) -> None:
        setup_env()
        self.api_key = get_gemini_api_key()
        self.model = get_gemini_model()
        self.api_url = (
            "https://generativelanguage.googleapis.com/v1beta/models/"
            f"{self.model}:generateContent"
        )
        with Path("./data/2026AnthemgHIPSBC.pdf").open("rb") as file:
            self.pdf_data = base64.standard_b64encode(file.read()).decode("utf-8")

    def answer_query(self, prompt: str) -> str:
        response = requests.post(
            self.api_url,
            headers={
                "Content-Type": "application/json",
                "x-goog-api-key": self.api_key,
            },
            json={
                "system_instruction": {
                    "parts": [{"text": SYSTEM_INSTRUCTION}]
                },
                "generationConfig": {"temperature": 0.2},
                "contents": [
                    {
                        "parts": [
                            {
                                "inline_data": {
                                    "mime_type": "application/pdf",
                                    "data": self.pdf_data,
                                }
                            },
                            {"text": prompt},
                        ]
                    }
                ],
            },
            timeout=120,
        )
        response.raise_for_status()
        result = response.json()
        return result["candidates"][0]["content"]["parts"][0]["text"].replace("$", r"\\$")

Overwriting agents.py


## 3.5. Test the Agent Class

Finally, import the `PolicyAgent` class you just created and test it with the same query to ensure it works as expected.

In [29]:
from agents import PolicyAgent

print("Running Health Insurance Policy Agent")
agent = PolicyAgent()
prompt = "How much would I pay for mental health therapy?"

response = agent.answer_query(prompt)
display(Markdown(response))

Running Health Insurance Policy Agent


If you need mental health, behavioral health, or substance abuse services:

*   **In-Network Provider:**
    *   Office visits: 10% coinsurance
    *   Other Outpatient: 10% coinsurance
    *   Inpatient services: 10% coinsurance
*   **Out-of-Network Provider:**
    *   Office visits: 30% coinsurance
    *   Other Outpatient: 30% coinsurance
    *   Inpatient services: 30% coinsurance

## 3.6. Resources

- [Gemini API Document Understanding](https://ai.google.dev/gemini-api/docs/document-processing)
- [Gemini API Models](https://ai.google.dev/gemini-api/docs/models)
- [Course Repo](https://github.com/holtskinner/A2AWalkthrough)